# Workshop 3 · 04 — Data Science auf unstrukturierten Daten ⭐⭐ (Aufgabe)

**Ziel:** Aus unstrukturiertem Text numerische Repräsentationen (**Embeddings**) erzeugen und darauf klassische Data-Science-Verfahren anwenden:
1. **Clustering** – thematisch verwandte Meldungen automatisch gruppieren.
2. **Anomalie-Erkennung** – auffällige Meldungen finden (IsolationForest auf Embeddings).
3. **Similarity-Suche** – „ähnliche Schäden finden" über semantische Ähnlichkeit statt Stichwortsuche.

**Ausgangslage:** Tabelle `asset_clean` aus Notebook 01.

> Zentrales Prinzip: **Unstrukturierte Texte → Vektoren (`ai.embed`) → klassisches ML.**

## Schritt 0 — Dokument bauen und embedden

**Aufgabe:**
1. Baue je Meldung ein "Dokument" aus `komponente`, `freitext_meldung`, `kunden_kommentar` (z.B. mit `concat_ws`).
2. Erzeuge mit `ai.embed(input_col='doc', output_col='vector')` einen Vektor pro Meldung.
3. Hole die relevanten Spalten inkl. `vector` mit `.toPandas()` in ein DataFrame `pdf`.

In [ ]:
# This code uses AI. Always review output for mistakes.
import synapse.ml.spark.aifunc as aifunc
from pyspark.sql.functions import concat_ws, col

base = spark.table('asset_clean')

# TODO: doc-Spalte aus komponente + freitext_meldung + kunden_kommentar bauen
# docs = base.withColumn('doc', concat_ws('. ', col('komponente'), col('freitext_meldung'), col('kunden_kommentar')))

# TODO: Embeddings erzeugen (Text -> Vektor)
# emb = docs.ai.embed(input_col='doc', output_col='vector')

# TODO: nach pandas holen
# pdf = emb.select('meldung_id', 'depot_norm', 'komponente', 'schweregrad',
#                  'freitext_meldung', 'vector').toPandas()
# print('Meldungen x Vektordim:', pdf.shape, '/', len(pdf['vector'].iloc[0]))

## 1) Clustering — Themen automatisch finden

**Aufgabe:** Staple die Vektoren zu einer Matrix `X` und wende `KMeans` (z.B. `k=4`) darauf an. Schreibe das Cluster-Label in `pdf['cluster']` und zeige die Cluster-Größen.

In [ ]:
import numpy as np
from sklearn.cluster import KMeans

# TODO: Vektoren zu Matrix X stapeln
# X = np.vstack(pdf['vector'].apply(lambda v: np.array(v, dtype=float)).values)

# TODO: KMeans clustern und Label speichern
# k = 4
# pdf['cluster'] = KMeans(n_clusters=k, n_init=10, random_state=0).fit_predict(X)
# display(pdf.groupby('cluster').size().reset_index(name='anzahl'))

### (Bonus) Cluster automatisch benennen
**Aufgabe:** Erzeuge je Cluster ein prägnantes Themen-Label aus ein paar Beispiel-Freitexten — mit der pandas-AI-Function `df.ai.generate_response(prompt=..., is_prompt_template=True)`.

In [ ]:
# This code uses AI. Always review output for mistakes.
import synapse.ml.aifunc as aifunc_pd
import pandas as pd

# TODO: je Cluster ein Label aus Beispieltexten erzeugen
# labels = {}
# for c in sorted(pdf['cluster'].unique()):
#     beispiele = ' || '.join(pdf[pdf.cluster == c]['freitext_meldung'].head(4).tolist())
#     one = pd.DataFrame({'beispiele': [beispiele]})
#     one['label'] = one.ai.generate_response(
#         prompt='Gib ein praegnantes Themen-Label auf Deutsch (max 4 Woerter) fuer: {beispiele}',
#         is_prompt_template=True)
#     labels[c] = one['label'].iloc[0]
# pdf['thema'] = pdf['cluster'].map(labels)
# display(pdf[['cluster', 'thema']].drop_duplicates().sort_values('cluster'))

## 2) Anomalie-Erkennung — ungewöhnliche Meldungen

**Aufgabe:** Wende `IsolationForest` (z.B. `contamination=0.15`) auf `X` an und markiere Anomalien in `pdf['anomalie']`. Zeige die auffälligen Meldungen.

In [ ]:
from sklearn.ensemble import IsolationForest

# TODO: IsolationForest auf Embeddings, Anomalien markieren (-1 == Anomalie)
# iso = IsolationForest(contamination=0.15, random_state=0)
# pdf['anomalie'] = (iso.fit_predict(X) == -1)
# print('Anomalien:', int(pdf['anomalie'].sum()), 'von', len(pdf))
# display(pdf[pdf['anomalie']][['meldung_id', 'komponente', 'schweregrad', 'freitext_meldung']])

## 3) Similarity-Suche — „finde ähnliche Schäden"

**Aufgabe:** Embedde eine Freitext-Anfrage und vergleiche sie per **Cosine-Ähnlichkeit** mit allen Meldungen (Matrix `X`). Gib die Top-Treffer zurück. *Tipp:* eine kleine `embed_query(q)`-Hilfsfunktion + `X @ qv / (||X|| * ||qv||)`.

In [ ]:
# This code uses AI. Always review output for mistakes.

# TODO: Hilfsfunktion, die eine Anfrage embeddet
# def embed_query(q):
#     row = (spark.createDataFrame([(q,)], ['doc'])
#            .ai.embed(input_col='doc', output_col='vector')
#            .collect()[0]['vector'])
#     return np.array(row, dtype=float)

# TODO: Cosine-Aehnlichkeit gegen X, Top-k zurueckgeben
# def find_similar(q, top_k=3):
#     qv = embed_query(q)
#     sims = X @ qv / (np.linalg.norm(X, axis=1) * np.linalg.norm(qv) + 1e-9)
#     idx = np.argsort(sims)[::-1][:top_k]
#     res = pdf.iloc[idx][['meldung_id', 'komponente', 'freitext_meldung']].copy()
#     res['aehnlichkeit'] = np.round(sims[idx], 3)
#     return res

# display(find_similar('Roststelle am Tuerrahmen, klemmende Tuer'))

## Schritt 4 — Ergebnisse persistieren
**Aufgabe:** Speichere die angereicherten Meldungen (`meldung_id`, `depot_norm`, `komponente`, `schweregrad`, `cluster`, `thema`, `anomalie`) als Tabelle `meldung_ml` — Grundlage für Power BI / Activator.

In [ ]:
# TODO: pandas -> Spark und als Tabelle meldung_ml speichern
# out = spark.createDataFrame(
#     pdf[['meldung_id', 'depot_norm', 'komponente', 'schweregrad', 'cluster', 'thema', 'anomalie']])
# out.write.mode('overwrite').option('overwriteSchema', 'true').saveAsTable('meldung_ml')
# display(out)

### ✅ Ergebnis-Check
Aus Text wurden Vektoren (`ai.embed`), darauf liefen **Clustering**, **Anomalie-Erkennung** und **semantische Suche** — Methoden, die ohne Embeddings nicht möglich wären. Ergebnis `meldung_ml`.

> **Bei großen Korpora:** Vektoren in **Eventhouse/KQL** legen und mit `series_cosine_similarity()` skalierend suchen statt im Treiber.